# Phase 6: Case Study Selection & Analysis

Selects representative samples across 7 case types and generates combined
multi-method explanation panels integrating Grad-CAM, Attention, SHAP, and LIME.

| Component | Description |
|---|---|
| Input | Artifacts from Phases 2-5 + prediction CSVs |
| Selection | 7 case types with multi-criteria ranking |
| Output | Combined figures (12×14, 300 DPI) + metadata + analysis |
| Architecture | Pure consumer — no model loading or inference |

**Case types:** correct, high_error, conflict, text_dominant, image_dominant, difficult, agreement

---
### STEP 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### STEP 2: Clone and install

In [ ]:
!rm -rf /content/SE365
!git clone -b xai-v2 https://github.com/lechihoang/SE365.git /content/SE365
%cd /content/SE365
!pip install -q -r requirements.txt

### STEP 3: Download and extract data

In [ ]:
!rm -rf ./data
!cp /content/drive/MyDrive/SE365/data.zip ./data.zip
!unzip -q data.zip
!rm data.zip
!ls -la ./data

### STEP 4: Configuration

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

DRIVE_ROOT   = '/content/drive/MyDrive/SE365'
PROJECT_ROOT = '/content/SE365'
EXP_ID       = 'EXP_060A_bestsequential_full_configuration'

EXP_DIR      = f'{DRIVE_ROOT}/experiments/{EXP_ID}'
XAI_DIR      = f'{EXP_DIR}/xai'
XAI_OUT_DIR  = f'{EXP_DIR}/xai/case_studies'
DATA_DIR     = f'{PROJECT_ROOT}/data/text'
IMAGE_DIR    = f'{PROJECT_ROOT}/data/image'

os.makedirs(XAI_OUT_DIR, exist_ok=True)
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'EXP_DIR      : {EXP_DIR}')
print(f'XAI_DIR      : {XAI_DIR}')
print(f'XAI_OUT_DIR  : {XAI_OUT_DIR}')

### STEP 5: Imports

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 5 — Imports')
print('='*60)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image as PILImage

from xai.config import (
    TARGET_NAMES, FACTOR_NAMES, DISPLAY_NAMES, NUM_TARGETS,
    DEFAULT_DPI, THESIS_DPI,
)
from xai.case_study import (
    CaseStudyRunner,
    check_sample_artifacts, load_shap_contribution,
    select_cases, create_combined_figure,
    generate_case_metadata, generate_analysis_text,
)

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 6: Check XAI Artifact Availability

Verify which phases have run and which samples have artifacts.

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 6 — Check Artifacts')
print('='*60)

# Check which phase directories exist
for phase in ['gradcam', 'attention', 'shap', 'lime']:
    phase_dir = os.path.join(XAI_DIR, phase)
    exists = os.path.isdir(phase_dir)
    if exists:
        samples = [d for d in os.listdir(phase_dir) if os.path.isdir(os.path.join(phase_dir, d)) and d.startswith('sample_')]
        print(f'  {phase:12s}: {len(samples)} samples')
    else:
        print(f'  {phase:12s}: NOT FOUND')

# Check prediction CSV
test_pred = os.path.join(EXP_DIR, 'test_predictions.csv')
val_pred = os.path.join(EXP_DIR, 'predictions.csv')
if os.path.isfile(test_pred):
    PRED_CSV = test_pred
    SPLIT = 'test'
elif os.path.isfile(val_pred):
    PRED_CSV = val_pred
    SPLIT = 'validation'
else:
    raise FileNotFoundError('No prediction CSV found!')

DATASET_CSV = os.path.join(DATA_DIR, f'{SPLIT}.csv' if SPLIT == 'test' else 'val.csv')
if not os.path.isfile(DATASET_CSV):
    DATASET_CSV = os.path.join(DATA_DIR, 'val.csv')

print(f'\n  Predictions : {PRED_CSV}')
print(f'  Dataset     : {DATASET_CSV}')
print(f'  Split       : {SPLIT}')
print(f'Done ({time.time()-t0:.1f}s)')

### STEP 7: Load Prediction Data

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 7 — Load Predictions + Artifact Diagnostics')
print('='*60)

pred_df = pd.read_csv(PRED_CSV)
dataset_df = pd.read_csv(DATASET_CSV)

# Add computed error columns
error_cols = [c for c in pred_df.columns if c.startswith('absolute_error_')]
if error_cols:
    pred_df['max_error'] = pred_df[error_cols].max(axis=1)
    pred_df['mean_error'] = pred_df[error_cols].mean(axis=1)

print(f'Predictions  : {len(pred_df)} samples')
print(f'Dataset      : {len(dataset_df)} samples')
print(f'Error range  : [{pred_df["mean_error"].min():.3f}, {pred_df["mean_error"].max():.3f}]')

# Artifact availability diagnostic
from xai.case_study import check_sample_artifacts
n_check = min(len(pred_df), len(dataset_df))
art_summary = {'gradcam': 0, 'attention': 0, 'shap': 0, 'lime': 0, 'complete': 0}
print(f'\n--- Artifact Availability (first {n_check} samples) ---')
print(f'{"sample_id":<16s} {"GradCAM":>8s} {"Attn":>8s} {"SHAP":>8s} {"LIME":>8s} {"Complete":>9s}')
print('-'*60)
for i in range(min(20, n_check)):
    sid = f'sample_{i:04d}'
    a = check_sample_artifacts(sid, XAI_DIR)
    for p in ['gradcam', 'attention', 'shap', 'lime']:
        art_summary[p] += int(a[p])
    if a['completeness'] == 1.0:
        art_summary['complete'] += 1
    if i < 20:
        gc = 'OK' if a['gradcam'] else '--'
        at = 'OK' if a['attention'] else '--'
        sh = 'OK' if a['shap'] else '--'
        lm = 'OK' if a['lime'] else '--'
        cp = f'{a["completeness"]:.0%}'
        print(f'{sid:<16s} {gc:>8s} {at:>8s} {sh:>8s} {lm:>8s} {cp:>9s}')

print(f'\nSummary: GradCAM={art_summary["gradcam"]}, Attention={art_summary["attention"]}, '
      f'SHAP={art_summary["shap"]}, LIME={art_summary["lime"]}, '
      f'Complete(4/4)={art_summary["complete"]}')

if art_summary['gradcam'] == 0 and art_summary['attention'] == 0:
    print('\nWARNING: No XAI artifacts found! Run Phases 2-5 first.')
    print('Phase 6 will only be able to select samples without explanations.')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 8: Run Full Case Study Pipeline

Uses `CaseStudyRunner` to:
1. Select cases across 7 types
2. Generate manifests
3. Create combined figures
4. Generate metadata + analysis
5. Create index + summary

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 8 — Run Case Study Pipeline')
print('='*60)

runner = CaseStudyRunner(
    exp_dir=EXP_DIR,
    dataset_csv=DATASET_CSV,
    image_dir=IMAGE_DIR,
    split=SPLIT,
)

results = runner.run()

print(f'\nPipeline complete ({time.time()-t0:.1f}s)')

### STEP 9: Display Selection Summary

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 9 — Selection Summary')
print('='*60)

selection = results.get('selection', {})
total = 0
print(f'\n{"Case Type":<18s} {"Count":>5s} {"Sample IDs"}')
print('-'*50)
for case_type in ['conflict', 'high_error', 'text_dominant', 'image_dominant', 'difficult', 'agreement', 'correct']:
    cases = selection.get(case_type, [])
    count = len(cases)
    total += count
    ids = ', '.join([str(c[0]) for c in cases[:5]])
    print(f'{case_type:<18s} {count:5d}   {ids}')
print(f'{"TOTAL":<18s} {total:5d}')

# Display manifest preview
manifest_path = os.path.join(XAI_OUT_DIR, 'sample_manifest_preview.md')
if os.path.isfile(manifest_path):
    with open(manifest_path, 'r', encoding='utf-8') as f:
        preview = f.read()
    print(f'\n--- Sample Manifest Preview (first 2000 chars) ---')
    print(preview[:2000])

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 10: Display Example Combined Figure

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 10 — Example Combined Figure')
print('='*60)

# Find the first generated combined figure
example_fig = None
for root, dirs, files in os.walk(XAI_OUT_DIR):
    for f in files:
        if f.startswith('combined_figure_') and f.endswith('.png'):
            example_fig = os.path.join(root, f)
            break
    if example_fig:
        break

if example_fig and os.path.isfile(example_fig):
    img = PILImage.open(example_fig)
    fig, ax = plt.subplots(1, 1, figsize=(12, 14))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Example Combined Figure', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f'Displayed: {example_fig}')
else:
    print('No combined figures found yet.')

print(f'Done ({time.time()-t0:.1f}s)')

### STEP 11: Validation Checks

In [ ]:
t0 = time.time()
print('='*60)
print('  Phase 6 — Step 11 — Validation')
print('='*60)

checks = []

# Check index CSV
index_path = os.path.join(XAI_OUT_DIR, 'case_study_index.csv')
index_ok = os.path.isfile(index_path)
if index_ok:
    index_df = pd.read_csv(index_path)
    checks.append(('Index CSV exists', True, f'{len(index_df)} rows'))
else:
    checks.append(('Index CSV exists', False, ''))

# Check manifest
manifest_csv = os.path.join(XAI_OUT_DIR, 'sample_manifest.csv')
checks.append(('Manifest CSV', os.path.isfile(manifest_csv), ''))

manifest_json = os.path.join(XAI_OUT_DIR, 'sample_manifest.json')
checks.append(('Manifest JSON', os.path.isfile(manifest_json), ''))

# Check selection log
sel_log = os.path.join(XAI_OUT_DIR, 'selection_log.json')
checks.append(('Selection log', os.path.isfile(sel_log), ''))

# Check summary
summary_md = os.path.join(XAI_OUT_DIR, 'case_study_summary.md')
checks.append(('Summary MD', os.path.isfile(summary_md), ''))

# Check case directories
case_dirs = [d for d in os.listdir(XAI_OUT_DIR) if d.startswith('case_') and os.path.isdir(os.path.join(XAI_OUT_DIR, d))]
checks.append(('Case directories', len(case_dirs) > 0, f'{len(case_dirs)} cases'))

# Check each case has metadata
meta_count = 0
for cd in case_dirs:
    if os.path.isfile(os.path.join(XAI_OUT_DIR, cd, 'metadata.json')):
        meta_count += 1
checks.append(('Metadata per case', meta_count == len(case_dirs), f'{meta_count}/{len(case_dirs)}'))

# Print results
all_passed = True
for desc, passed, extra in checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed:
        all_passed = False
    extra_str = f'  ({extra})' if extra else ''
    print(f'  [{s:6s}] {desc}{extra_str}')

print(f'\nDone ({time.time()-t0:.1f}s)')

### STEP 12: Final Summary

In [ ]:
print('='*60)
print('  PHASE 6 CASE STUDY — FINAL SUMMARY')
print('='*60)

artifact_count = sum(len(f) for _, _, f in os.walk(XAI_OUT_DIR))

print(f'  Experiment      : {EXP_ID}')
print(f'  Split           : {SPLIT}')
print(f'  Case types      : {len([k for k, v in selection.items() if v])}')
print(f'  Total cases     : {total}')
print(f'  Case directories: {len(case_dirs)}')
print(f'  Total artifacts : {artifact_count}')
print(f'  Output dir      : {XAI_OUT_DIR}')
print()

final_checks = [
    ('Selection completed',    total > 0),
    ('Manifests generated',    os.path.isfile(manifest_csv)),
    ('Index CSV generated',    index_ok),
    ('Summary generated',      os.path.isfile(summary_md)),
    ('Selection log saved',    os.path.isfile(sel_log)),
    ('Case dirs created',      len(case_dirs) > 0),
    ('All metadata present',   meta_count == len(case_dirs)),
]

all_ok = True
for desc, passed in final_checks:
    s = 'PASSED' if passed else 'FAILED'
    if not passed: all_ok = False
    print(f'  [{s:6s}] {desc}')

print()
print('  This phase is a PURE CONSUMER of Phases 2-5 artifacts.')
print('  Combined figures integrate Grad-CAM + Attention + SHAP + LIME.')
print('  Use case_study_index.csv for thesis Results chapter.')

print('='*60)
if all_ok:
    print('  All checks PASSED. Phase 6 complete.')
else:
    print('  Some checks FAILED. Review output above.')
print('='*60)